# Exploratory analysis: what drives a block group's value

Before the store and the endpoint, the data itself. Which census attributes
move median house value, and how strongly?

In [ ]:
import matplotlib.pyplot as plt
from california_house_pricing.prepare_data import load_frame

frame = load_frame()
frame.describe().round(2)

`median_house_value` is in $100k units and top-coded at 5.0 — the flat
top of the histogram is the census cap, not a modelling artifact.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
frame["median_house_value"].hist(bins=60, ax=axes[0])
axes[0].set_title("label distribution")
frame["median_income"].hist(bins=60, ax=axes[1])
axes[1].set_title("median_income")
plt.tight_layout()

In [ ]:
frame.corr(numeric_only=True)["median_house_value"].drop(
    "median_house_value"
).sort_values().plot.barh(figsize=(7, 4), title="correlation with value")

**Income dominates** — `median_income` is by far the strongest single
predictor; location (`latitude`/`longitude`) carries the rest. Tree models
pick up the geographic interaction income alone misses.

In [ ]:
frame.plot.scatter(
    x="longitude",
    y="latitude",
    c="median_house_value",
    cmap="viridis",
    s=4,
    figsize=(6, 7),
    title="value by geography",
)

The coastline is unmistakable — the model must see both position and
income. That joint view is what LightGBM's leaf splits buy us.